In [2]:
import numpy as np

import scanpy as sc
from collections import Counter
import gc
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
import os

In [ ]:
sample_list_st = [
  "P1","P2","P3","P4","P5","P6","P7","P8","P9","P10","P11","P12","P13","P14","P15","P16","P26","P27","P30","P32","P33","P34","P35","P36","P37","P38","P39",
  "P40_1","P40_2","P42","P43_1","P43_2","P44","P45","P46","P47","P48","P49"
]

In [6]:
def calculate_min_distances(coord_df, celltype_anchor, celltype_target):
    """
    Calculate the shortest distance from each anchor cell to the target cell types
    
    Parameters:
    coord_df -- DataFrame containing cell coordinates and types; required columns: ['x', 'y', 'celltype'], indexed by cell_id
    celltype_anchor -- String specifying the anchor cell type
    celltype_target -- List of one or more target cell types
    
    Returns:
    DataFrame indexed by anchor cell_id, with target cell types as columns and the nearest distance to each cell type as values
    """
    # Verify that the input columns exist
    required_columns = ['x', 'y', 'celltype']
    if not all(col in coord_df.columns for col in required_columns):
        raise ValueError("Input DataFrame is missing required columns: x, y, celltype")
    
    # Extract anchor cells
    anchor_cells = coord_df[coord_df['celltype'] == celltype_anchor]
    if anchor_cells.empty:
        raise ValueError(f"Anchor cell type not found in the data: {celltype_anchor}")
    
    # Prepare the results DataFrame
    result_df = pd.DataFrame(
        index=anchor_cells.index,
        columns=celltype_target,
        dtype=float
    )
    
    # Calculate the nearest distance for each target cell type
    for target_type in celltype_target:
        # Extract cells of the current target type
        target_cells = coord_df[coord_df['celltype'] == target_type]
        
        if target_cells.empty:
            result_df[target_type] = np.nan
            continue
        
        # Build a KDTree for the target cells
        coords_target = target_cells[['x', 'y']].values
        tree = cKDTree(coords_target)
        
        # Calculate the minimum distance from anchor cells to target cells
        anchor_coords = anchor_cells[['x', 'y']].values
        distances, _ = tree.query(anchor_coords, k=1)
        
        # Store results
        result_df[target_type] = distances
    
    return result_df

# Calculate distances from DCIS and IDC epithelial cells to T cells and B cells

In [4]:
distance_epi_t_mean_all = pd.DataFrame()
distance_epi_b_mean_all = pd.DataFrame()
distance_epi_macro_mean_all = pd.DataFrame()

In [ ]:
for spID in sample_list_st:
    # Load the coordinate file and convert all units to micrometers
    sp_coords = pd.read_csv(f'E:/Projects/BRCA/results/ST/Niche2607/{spID}/coord_{spID}.csv', index_col = 0)
    npixel80 = pd.read_csv(f'E:/Projects/BRCA/results/ST/Niche2607/{spID}/wanted_r.txt', sep="\t", header=None)
    n_um1pixel = 80 / npixel80.values[0,0]
    sp_coords = sp_coords * n_um1pixel
    
    # Add region information
    sp_celltype = pd.read_csv(f'E:/Projects/BRCA/results/ST/Niche2607/{spID}/celltype_{spID}.csv', index_col=0)
    sp_coords['celltype'] = sp_celltype['celltype'].reindex(sp_coords.index)
    sp_region = pd.read_csv(f'E:/Projects/BRCA/results/ST/cluster_ljy/all_sample_region/{spID}_region.csv', index_col=0)
    sp_coords['region'] = sp_region.reindex(sp_coords.index)
    sp_coords = sp_coords.fillna('undefined')
    sp_coords = sp_coords.loc[~((sp_coords['celltype'] == 'Epithelial') & (sp_coords['region'] == 'undefined'))]
    sp_coords.loc[sp_coords['celltype'] == 'Epithelial', 'celltype'] = sp_coords.loc[sp_coords['celltype'] == 'Epithelial', 'region'] + '_' + sp_coords.loc[sp_coords['celltype'] == 'Epithelial', 'celltype']
    
    if Counter(sp_coords['celltype'])['DCIS_Epithelial'] < 2000:
        print("number of DCIS epi cells too low")
        continue
    if Counter(sp_coords['celltype'])['IDC_Epithelial'] < 2000:
        print("number of IDC epi cells too low")
        continue
    output_dir = f"E:/Projects/BRCA/results/ST/distance/{spID}/"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    if Counter(sp_celltype['celltype'])['T'] > 200:
        distance_idc_t = calculate_min_distances(coord_df=sp_coords, celltype_anchor="IDC_Epithelial", celltype_target=['T'])
        distance_idc_t.to_csv(output_dir + 'distance_IDC_epi_to_T.csv')
        distance_idc_t_mean = distance_idc_t.mean(axis = 0)
        distance_idc_t_mean = pd.DataFrame(distance_idc_t_mean)
        distance_idc_t_mean = distance_idc_t_mean.T
        distance_idc_t_mean['sample'] = spID
        distance_idc_t_mean['region'] = 'IDC'
        distance_epi_t_mean_all = pd.concat([distance_epi_t_mean_all, distance_idc_t_mean])
    if Counter(sp_celltype['celltype'])['T'] > 200:
        distance_dcis_t = calculate_min_distances(coord_df=sp_coords, celltype_anchor="DCIS_Epithelial", celltype_target=['T'])
        distance_dcis_t.to_csv(output_dir + 'distance_DCIS_epi_to_T.csv')
        distance_dcis_t_mean = distance_dcis_t.mean(axis = 0)
        distance_dcis_t_mean = pd.DataFrame(distance_dcis_t_mean)
        distance_dcis_t_mean = distance_dcis_t_mean.T
        distance_dcis_t_mean['sample'] = spID
        distance_dcis_t_mean['region'] = 'DCIS'
        distance_epi_t_mean_all = pd.concat([distance_epi_t_mean_all, distance_dcis_t_mean])
        
        
    if Counter(sp_celltype['celltype'])['B'] > 200:
        distance_idc_b = calculate_min_distances(coord_df=sp_coords, celltype_anchor="IDC_Epithelial", celltype_target=['B'])
        distance_idc_b.to_csv(output_dir + 'distance_IDC_epi_to_B.csv')
        distance_idc_b_mean = distance_idc_b.mean(axis = 0)
        distance_idc_b_mean = pd.DataFrame(distance_idc_b_mean)
        distance_idc_b_mean = distance_idc_b_mean.T
        distance_idc_b_mean['sample'] = spID
        distance_idc_b_mean['region'] = 'IDC'
        distance_epi_b_mean_all = pd.concat([distance_epi_b_mean_all, distance_idc_b_mean])
    if Counter(sp_celltype['celltype'])['B'] > 200:
        distance_dcis_b = calculate_min_distances(coord_df=sp_coords, celltype_anchor="DCIS_Epithelial", celltype_target=['B'])
        distance_dcis_b.to_csv(output_dir + 'distance_DCIS_epi_to_B.csv')
        distance_dcis_b_mean = distance_dcis_b.mean(axis = 0)
        distance_dcis_b_mean = pd.DataFrame(distance_dcis_b_mean)
        distance_dcis_b_mean = distance_dcis_b_mean.T
        distance_dcis_b_mean['sample'] = spID
        distance_dcis_b_mean['region'] = 'DCIS'
        distance_epi_b_mean_all = pd.concat([distance_epi_b_mean_all, distance_dcis_b_mean])
        
        
    if Counter(sp_celltype['celltype'])['Macrophages'] > 1000:
        distance_idc_macro = calculate_min_distances(coord_df=sp_coords, celltype_anchor="IDC_Epithelial", celltype_target=['Macrophages'])
        distance_idc_macro.to_csv(output_dir + 'distance_IDC_epi_to_Macro.csv')
        distance_idc_macro_mean = distance_idc_macro.mean(axis = 0)
        distance_idc_macro_mean = pd.DataFrame(distance_idc_macro_mean)
        distance_idc_macro_mean = distance_idc_macro_mean.T
        distance_idc_macro_mean['sample'] = spID
        distance_idc_macro_mean['region'] = 'IDC'
        distance_epi_macro_mean_all = pd.concat([distance_epi_macro_mean_all, distance_idc_macro_mean])
    if Counter(sp_celltype['celltype'])['Macrophages'] > 1000:
        distance_dcis_macro = calculate_min_distances(coord_df=sp_coords, celltype_anchor="DCIS_Epithelial", celltype_target=['Macrophages'])
        distance_dcis_macro.to_csv(output_dir + 'distance_DCIS_epi_to_Macro.csv')
        distance_dcis_macro_mean = distance_dcis_macro.mean(axis = 0)
        distance_dcis_macro_mean = pd.DataFrame(distance_dcis_macro_mean)
        distance_dcis_macro_mean = distance_dcis_macro_mean.T
        distance_dcis_macro_mean['sample'] = spID
        distance_dcis_macro_mean['region'] = 'DCIS'
        distance_epi_macro_mean_all = pd.concat([distance_epi_macro_mean_all, distance_dcis_macro_mean])

In [9]:
distance_epi_b_mean_all.to_csv("E:/Projects/BRCA/results/ST/distance/distance_epi_to_b_mean_2609.csv")

In [10]:
distance_epi_t_mean_all.to_csv("E:/Projects/BRCA/results/ST/distance/distance_epi_to_t_mean_2609.csv")

In [11]:
distance_epi_macro_mean_all.to_csv("E:/Projects/BRCA/results/ST/distance/distance_epi_to_macro_mean_2609.csv")

# Calculate distances from DCIS and IDC epithelial cells to macrophage subtypes

In [3]:
macro_subcelltype = pd.read_csv("E:/Projects/BRCA/results/ST/Macrophage/mye_sub_celltype.csv", index_col=0)

In [7]:
macro_subcelltype = macro_subcelltype.loc[macro_subcelltype['sub_celltype'].isin(['Macro_C1','Macro_C2','Macro_C3','Macro_C4'])]

In [23]:
distance_epi_lam_mean_all = pd.DataFrame()
distance_epi_macro_cxcl10_mean_all = pd.DataFrame()
distance_epi_macro_folr2_mean_all = pd.DataFrame()
distance_epi_macro_mki67_mean_all = pd.DataFrame()

In [ ]:
for spID in sample_list_st:
    # Load the coordinate file and convert all units to micrometers
    print(spID)
    sp_coords = pd.read_csv(f'E:/Projects/BRCA/results/ST/Niche2607/{spID}/coord_{spID}.csv', index_col = 0)
    npixel80 = pd.read_csv(f'E:/Projects/BRCA/results/ST/Niche2607/{spID}/wanted_r.txt', sep="\t", header=None)
    n_um1pixel = 80 / npixel80.values[0,0]
    sp_coords = sp_coords * n_um1pixel
    # Add cell type information
    sp_celltype = pd.read_csv(f'E:/Projects/BRCA/results/ST/Niche2607/{spID}/celltype_{spID}.csv', index_col=0)
    sp_coords['celltype'] = sp_celltype['celltype'].reindex(sp_coords.index)
    # Update macrophage subtypes
    sp_coords.loc[sp_coords.index.isin(macro_subcelltype.index.to_list()),'celltype'] = macro_subcelltype.loc[sp_coords.index[sp_coords.index.isin(macro_subcelltype.index.to_list())], "sub_celltype"]
    sp_region = pd.read_csv(f'E:/Projects/BRCA/results/ST/cluster_ljy/all_sample_region/{spID}_region.csv', index_col=0)
    sp_coords['region'] = sp_region.reindex(sp_coords.index)
    sp_coords = sp_coords.fillna('undefined')
    sp_coords = sp_coords.loc[~((sp_coords['celltype'] == 'Epithelial') & (sp_coords['region'] == 'undefined'))]
    sp_coords.loc[sp_coords['celltype'] == 'Epithelial', 'celltype'] = sp_coords.loc[sp_coords['celltype'] == 'Epithelial', 'region'] + '_' + sp_coords.loc[sp_coords['celltype'] == 'Epithelial', 'celltype']
    
    if Counter(sp_coords['celltype'])['DCIS_Epithelial'] < 2000:
        print("number of DCIS epi cells too low")
        continue
    if Counter(sp_coords['celltype'])['IDC_Epithelial'] < 2000:
        print("number of IDC epi cells too low")
        continue
    output_dir = f"E:/Projects/BRCA/results/ST/distance/{spID}/"
    
    # Calculate distances between epithelial cells and LAM
    if Counter(sp_coords['celltype'])['Macro_C3'] > 500:
        distance_idc_lam = calculate_min_distances(coord_df=sp_coords, celltype_anchor="IDC_Epithelial", celltype_target=['Macro_C3'])
        distance_idc_lam.to_csv(output_dir + 'distance_IDC_epi_to_Macro_C3.csv')
        distance_idc_lam_mean = distance_idc_lam.mean(axis = 0)
        distance_idc_lam_mean = pd.DataFrame(distance_idc_lam_mean)
        distance_idc_lam_mean = distance_idc_lam_mean.T
        distance_idc_lam_mean['sample'] = spID
        distance_idc_lam_mean['region'] = 'IDC'
        distance_epi_lam_mean_all = pd.concat([distance_epi_lam_mean_all, distance_idc_lam_mean])
    if Counter(sp_coords['celltype'])['Macro_C3'] > 500:
        distance_dcis_lam = calculate_min_distances(coord_df=sp_coords, celltype_anchor="DCIS_Epithelial", celltype_target=['Macro_C3'])
        distance_dcis_lam.to_csv(output_dir + 'distance_DCIS_epi_to_Macro_C3.csv')
        distance_dcis_lam_mean = distance_dcis_lam.mean(axis = 0)
        distance_dcis_lam_mean = pd.DataFrame(distance_dcis_lam_mean)
        distance_dcis_lam_mean = distance_dcis_lam_mean.T
        distance_dcis_lam_mean['sample'] = spID
        distance_dcis_lam_mean['region'] = 'DCIS'
        distance_epi_lam_mean_all = pd.concat([distance_epi_lam_mean_all, distance_dcis_lam_mean])
    # Calculate distances between epithelial cells and Macro CXCL10+
    if Counter(sp_coords['celltype'])['Macro_C1'] > 500:
        distance_idc_cxcl10 = calculate_min_distances(coord_df=sp_coords, celltype_anchor="IDC_Epithelial", celltype_target=['Macro_C1'])
        distance_idc_cxcl10.to_csv(output_dir + 'distance_IDC_epi_to_macro_Macro_C1.csv')
        distance_idc_cxcl10_mean = distance_idc_cxcl10.mean(axis = 0)
        distance_idc_cxcl10_mean = pd.DataFrame(distance_idc_cxcl10_mean)
        distance_idc_cxcl10_mean = distance_idc_cxcl10_mean.T
        distance_idc_cxcl10_mean['sample'] = spID
        distance_idc_cxcl10_mean['region'] = 'IDC'
        distance_epi_macro_cxcl10_mean_all = pd.concat([distance_epi_macro_cxcl10_mean_all, distance_idc_cxcl10_mean])
    if Counter(sp_coords['celltype'])['Macro_C1'] > 500:
        distance_dcis_cxcl10 = calculate_min_distances(coord_df=sp_coords, celltype_anchor="DCIS_Epithelial", celltype_target=['Macro_C1'])
        distance_dcis_cxcl10.to_csv(output_dir + 'distance_DCIS_epi_to_macro_Macro_C1.csv')
        distance_dcis_cxcl10_mean = distance_dcis_cxcl10.mean(axis = 0)
        distance_dcis_cxcl10_mean = pd.DataFrame(distance_dcis_cxcl10_mean)
        distance_dcis_cxcl10_mean = distance_dcis_cxcl10_mean.T
        distance_dcis_cxcl10_mean['sample'] = spID
        distance_dcis_cxcl10_mean['region'] = 'DCIS'
        distance_epi_macro_cxcl10_mean_all = pd.concat([distance_epi_macro_cxcl10_mean_all, distance_dcis_cxcl10_mean])
    # Calculate distances between epithelial cells and Macro_C2 FOLR2+
    if Counter(sp_coords['celltype'])['Macro_C2'] > 100:
        distance_idc_folr2 = calculate_min_distances(coord_df=sp_coords, celltype_anchor="IDC_Epithelial", celltype_target=['Macro_C2'])
        distance_idc_folr2.to_csv(output_dir + 'distance_IDC_epi_to_macro_Macro_C2.csv')
        distance_idc_folr2_mean = distance_idc_folr2.mean(axis = 0)
        distance_idc_folr2_mean = pd.DataFrame(distance_idc_folr2_mean)
        distance_idc_folr2_mean = distance_idc_folr2_mean.T
        distance_idc_folr2_mean['sample'] = spID
        distance_idc_folr2_mean['region'] = 'IDC'
        distance_epi_macro_folr2_mean_all = pd.concat([distance_epi_macro_folr2_mean_all, distance_idc_folr2_mean])
    if Counter(sp_coords['celltype'])['Macro_C2'] > 100:
        distance_dcis_folr2 = calculate_min_distances(coord_df=sp_coords, celltype_anchor="DCIS_Epithelial", celltype_target=['Macro_C2'])
        distance_dcis_folr2.to_csv(output_dir + 'distance_DCIS_epi_to_macro_Macro_C2.csv')
        distance_dcis_folr2_mean = distance_dcis_folr2.mean(axis = 0)
        distance_dcis_folr2_mean = pd.DataFrame(distance_dcis_folr2_mean)
        distance_dcis_folr2_mean = distance_dcis_folr2_mean.T
        distance_dcis_folr2_mean['sample'] = spID
        distance_dcis_folr2_mean['region'] = 'DCIS'
        distance_epi_macro_folr2_mean_all = pd.concat([distance_epi_macro_folr2_mean_all, distance_dcis_folr2_mean])
    # Calculate distances between epithelial cells and Macro_C4 MKI67+
    if Counter(sp_coords['celltype'])['Macro_C4'] > 100:
        distance_idc_mki67 = calculate_min_distances(coord_df=sp_coords, celltype_anchor="IDC_Epithelial", celltype_target=['Macro_C4'])
        distance_idc_mki67.to_csv(output_dir + 'distance_IDC_epi_to_macro_Macro_C4.csv')
        distance_idc_mki67_mean = distance_idc_mki67.mean(axis = 0)
        distance_idc_mki67_mean = pd.DataFrame(distance_idc_mki67_mean)
        distance_idc_mki67_mean = distance_idc_mki67_mean.T
        distance_idc_mki67_mean['sample'] = spID
        distance_idc_mki67_mean['region'] = 'IDC'
        distance_epi_macro_mki67_mean_all = pd.concat([distance_epi_macro_mki67_mean_all, distance_idc_mki67_mean])
    if Counter(sp_coords['celltype'])['Macro_C4'] > 100:
        distance_dcis_mki67 = calculate_min_distances(coord_df=sp_coords, celltype_anchor="DCIS_Epithelial", celltype_target=['Macro_C4'])
        distance_dcis_mki67.to_csv(output_dir + 'distance_DCIS_epi_to_macro_Macro_C4.csv')
        distance_dcis_mki67_mean = distance_dcis_mki67.mean(axis = 0)
        distance_dcis_mki67_mean = pd.DataFrame(distance_dcis_mki67_mean)
        distance_dcis_mki67_mean = distance_dcis_mki67_mean.T
        distance_dcis_mki67_mean['sample'] = spID
        distance_dcis_mki67_mean['region'] = 'DCIS'
        distance_epi_macro_mki67_mean_all = pd.concat([distance_epi_macro_mki67_mean_all, distance_dcis_mki67_mean])

In [ ]:
distance_epi_lam_mean_all = pd.DataFrame()
distance_epi_macro_cxcl10_mean_all = pd.DataFrame()
distance_epi_macro_folr2_mean_all = pd.DataFrame()
distance_epi_macro_mki67_mean_all = pd.DataFrame()

In [25]:
distance_epi_lam_mean_all.to_csv('E:/Projects/BRCA/results/ST/distance/distance_epi_to_Macro_C3_mean.csv')

In [26]:
distance_epi_macro_cxcl10_mean_all.to_csv('E:/Projects/BRCA/results/ST/distance/distance_epi_to_Macro_C1_mean.csv')

In [28]:
distance_epi_macro_folr2_mean_all.to_csv('E:/Projects/BRCA/results/ST/distance/distance_epi_to_Macro_C2_mean.csv')

In [30]:
distance_epi_macro_mki67_mean_all.to_csv('E:/Projects/BRCA/results/ST/distance/distance_epi_to_Macro_C4_mean.csv')